In [1]:
import os, json, time, random
import pandas as pd
from dotenv import load_dotenv
import anthropic

load_dotenv()  # reads your secret key from the .env file
client = anthropic.Anthropic()  # connects to Claude using that key
MODEL = "claude-haiku-5-5"
print("Ready! Key found:", os.getenv("ANTHROPIC_API_KEY", "")[:10] + "...")

Ready! Key found: sk-ant-usr...


In [2]:
from datasets import load_dataset

ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df = ds["train"].to_pandas()[["instruction", "category", "intent"]]
print("Rows:", len(df))
df.head()

/Library/Frameworks/Python.framework/Versions/3.15/lib/python3.15/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 26872/26872 [00:00<00:00, 416839.89 examples/s]

Rows: 26872


,instruction,category,intent
0,question about cancelling order {{Order Number}},ORDER,cancel_order
1,i have a question about cancelling oorder {{Or...,ORDER,cancel_order
2,i need help cancelling puchase {{Order Number}},ORDER,cancel_order
3,I need to cancel purchase {{Order Number}},ORDER,cancel_order
4,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order


In [3]:
print("Number of intents:", df["intent"].nunique())
df["intent"].value_counts()

Number of intents: 27


intent
check_invoice               1000
complaint                   1000
contact_customer_service    1000
edit_account                1000
switch_account              1000
check_payment_methods        999
contact_human_agent          999
delivery_period              999
get_invoice                  999
newsletter_subscription      999
payment_issue                999
registration_problems        999
cancel_order                 998
place_order                  998
track_refund                 998
change_order                 997
check_refund_policy          997
create_account               997
get_refund                   997
review                       997
set_up_shipping_address      997
delete_account               995
delivery_options             995
recover_password             995
track_order                  995
change_shipping_address      973
check_cancellation_fee       950
Name: count, dtype: int64

In [4]:
random.seed(42)
test = (df.groupby("intent").sample(n=19, random_state=42)
          .sample(frac=1, random_state=42)
          .head(500)
          .reset_index(drop=True))
rest = df.drop(df.index[df["instruction"].isin(test["instruction"])])
examples = rest.groupby("intent").sample(n=2, random_state=7)
INTENTS = sorted(df["intent"].unique())
print("Test messages:", len(test))
print("Example messages for the prompt:", len(examples))
test.head()

Test messages: 500
Example messages for the prompt: 54


,instruction,category,intent
0,I don't know how to get compensations,REFUND,get_refund
1,I need assistance checking my compensation status,REFUND,track_refund
2,need to set my new shipping address up,SHIPPING,set_up_shipping_address
3,can uhelp me speaking with customer service,CONTACT,contact_customer_service
4,I expect a fucking compensation of {{Currency ...,REFUND,track_refund


In [5]:
RULES = [
    ("track_refund", ["refund status", "track my refund", "track the refund", "where is my refund", "check my refund", "refund going"]),
    ("check_refund_policy", ["refund policy", "money back guarantee", "reimbursement policy"]),
    ("get_refund", ["refund", "reimburse", "money back"]),
    ("check_cancellation_fee", ["cancellation fee", "cancellation charge", "early termination", "penalty"]),
    ("cancel_order", ["cancel"]),
    ("track_order", ["track", "where is my order", "order status", "status of my order"]),
    ("change_order", ["change my order", "modify my order", "edit my order", "update my order", "add an item", "remove an item", "change order", "modify order"]),
    ("place_order", ["place an order", "buy", "purchase", "make an order", "order some", "order a"]),
    ("change_shipping_address", ["change my address", "change the address", "change shipping", "update my address", "update shipping", "edit my address", "modify my address"]),
    ("set_up_shipping_address", ["set up", "new address", "add an address", "add a shipping", "submit a new", "setup"]),
    ("delivery_period", ["how long", "when will", "delivery time", "arrive", "eta", "how soon"]),
    ("delivery_options", ["delivery option", "shipping option", "shipping method", "delivery method", "ship to"]),
    ("get_invoice", ["get my invoice", "download", "send me the invoice", "invoice copy", "get invoice", "get the invoice", "need the invoice", "need my invoice"]),
    ("check_invoice", ["invoice", "bill"]),
    ("payment_issue", ["payment issue", "payment problem", "can't pay", "cannot pay", "payment failed", "error with payment", "trouble paying", "problem paying", "issue paying"]),
    ("check_payment_methods", ["payment method", "payment option", "pay with", "pay by", "accept"]),
    ("recover_password", ["password", "forgot", "reset"]),
    ("registration_problems", ["registration", "sign up problem", "signup", "register", "can't sign up"]),
    ("delete_account", ["delete", "close my account", "remove my account", "deactivate", "terminate my account"]),
    ("switch_account", ["switch", "another account", "change account", "change to"]),
    ("edit_account", ["edit", "update my account", "modify my account", "change my account", "update my profile", "profile"]),
    ("create_account", ["create", "open an account", "new account", "sign up", "make an account"]),
    ("newsletter_subscription", ["newsletter", "subscribe", "unsubscribe", "mailing list"]),
    ("contact_human_agent", ["human", "agent", "real person", "representative", "operator"]),
    ("contact_customer_service", ["customer service", "customer support", "contact", "call", "phone", "email", "hours"]),
    ("complaint", ["complain", "complaint", "dissatisfied", "unhappy", "angry", "terrible", "worst"]),
    ("review", ["review", "feedback", "rate", "opinion", "comment"]),
]

def keyword_guess(text):
    t = text.lower()
    for intent, words in RULES:
        if any(w in t for w in words):
            return intent
    return "unknown"

test["keyword_pred"] = test["instruction"].apply(keyword_guess)
baseline_acc = (test["keyword_pred"] == test["intent"]).mean() * 100
print(f"Keyword baseline accuracy: {baseline_acc:.1f}%   <-- this is your [Y]%")

Keyword baseline accuracy: 44.0%   <-- this is your [Y]%


In [6]:
MEANINGS = {
    "cancel_order": "wants to cancel an order", "change_order": "wants to change items in an existing order",
    "change_shipping_address": "wants to change an existing delivery address", "check_cancellation_fee": "asks about cancellation or early termination fees",
    "check_invoice": "wants to look at or check an invoice or bill", "check_payment_methods": "asks which payment methods are accepted",
    "check_refund_policy": "asks about the refund policy", "complaint": "wants to complain or is unhappy",
    "contact_customer_service": "wants contact details or hours of customer service", "contact_human_agent": "wants to talk to a real person",
    "create_account": "wants to open a new account", "delete_account": "wants to close or delete an account",
    "delivery_options": "asks what delivery or shipping options exist", "delivery_period": "asks how long delivery will take",
    "edit_account": "wants to edit account or profile details", "get_invoice": "wants a copy of an invoice sent or downloaded",
    "get_refund": "wants money back", "newsletter_subscription": "wants to subscribe or unsubscribe from a newsletter",
    "payment_issue": "has a problem making a payment", "place_order": "wants to buy or place a new order",
    "recover_password": "forgot or wants to reset a password", "registration_problems": "has trouble signing up",
    "review": "wants to leave a review or feedback", "set_up_shipping_address": "wants to add a new delivery address",
    "switch_account": "wants to switch to another account", "track_order": "wants to know where an order is",
    "track_refund": "wants to know the status of a refund",
}

lines = [f"- {i}: {MEANINGS.get(i, '')}" for i in INTENTS]
shots = [f'Message: "{r.instruction}" -> {r.intent}' for r in examples.itertuples()]

SYSTEM_PROMPT = f"""You sort customer support messages for an online shop.

Choose exactly ONE intent from this list:
{chr(10).join(lines)}

Also give:
- urgency: "high" if the customer is blocked, losing money or very upset; "medium" if they need help soon; "low" if it is a general question.
- confidence: a number from 0 to 1 for how sure you are about the intent.

Solved examples:
{chr(10).join(shots)}
"""

SCHEMA = {
    "type": "object",
    "properties": {
        "intent": {"type": "string", "enum": INTENTS},
        "urgency": {"type": "string", "enum": ["high", "medium", "low"]},
        "confidence": {"type": "number"},
    },
    "required": ["intent", "urgency", "confidence"],
    "additionalProperties": False,
}
print(SYSTEM_PROMPT[:1500])

You sort customer support messages for an online shop.

Choose exactly ONE intent from this list:
- cancel_order: wants to cancel an order
- change_order: wants to change items in an existing order
- change_shipping_address: wants to change an existing delivery address
- check_cancellation_fee: asks about cancellation or early termination fees
- check_invoice: wants to look at or check an invoice or bill
- check_payment_methods: asks which payment methods are accepted
- check_refund_policy: asks about the refund policy
- complaint: wants to complain or is unhappy
- contact_customer_service: wants contact details or hours of customer service
- contact_human_agent: wants to talk to a real person
- create_account: wants to open a new account
- delete_account: wants to close or delete an account
- delivery_options: asks what delivery or shipping options exist
- delivery_period: asks how long delivery will take
- edit_account: wants to edit account or profile details
- get_invoice: wants a 

In [7]:
def classify(text):
    for attempt in range(3):
        try:
            resp = client.messages.create(
                model=MODEL,
                max_tokens=300,
                system=SYSTEM_PROMPT,
                thinking={"type": "disabled"},
                output_config={"effort": "low",
                               "format": {"type": "json_schema", "schema": SCHEMA}},
                messages=[{"role": "user", "content": text}],
            )
            raw = "".join(b.text for b in resp.content if b.type == "text")
            out = json.loads(raw)
            out["input_tokens"] = resp.usage.input_tokens
            out["output_tokens"] = resp.usage.output_tokens
            return out
        except Exception as e:
            print("  problem, trying again:", str(e)[:120])
            time.sleep(3)
    return {"intent": "error", "urgency": "error", "confidence": 0,
            "input_tokens": 0, "output_tokens": 0}

print(classify("hey i ordered yesterday and still nothing, where is it???"))

{'intent': 'track_order', 'urgency': 'medium', 'confidence': 0.9, 'input_tokens': 2629, 'output_tokens': 27}


In [8]:
results = []
for n, text in enumerate(test["instruction"], start=1):
    results.append(classify(text))
    if n % 25 == 0:
        print(f"Done {n} of {len(test)}")

res = pd.DataFrame(results)
test["llm_pred"] = res["intent"].values
test["urgency"] = res["urgency"].values
test["confidence"] = res["confidence"].values
test["input_tokens"] = res["input_tokens"].values
test["output_tokens"] = res["output_tokens"].values
test.to_csv("results_project1.csv", index=False)
print("Saved results_project1.csv")

Done 25 of 500
Done 50 of 500
Done 75 of 500
Done 100 of 500
Done 125 of 500
Done 150 of 500
Done 175 of 500
Done 200 of 500
Done 225 of 500
Done 250 of 500
Done 275 of 500
Done 300 of 500
Done 325 of 500
Done 350 of 500
Done 375 of 500
Done 400 of 500
Done 425 of 500
Done 450 of 500
Done 475 of 500
Done 500 of 500
Saved results_project1.csv


In [9]:
llm_acc = (test["llm_pred"] == test["intent"]).mean() * 100
print(f"Claude accuracy:           {llm_acc:.1f}%   <-- this is your [X]%")
print(f"Keyword baseline accuracy: {baseline_acc:.1f}%   <-- this is your [Y]%")

Claude accuracy:           93.6%   <-- this is your [X]%
Keyword baseline accuracy: 44.0%   <-- this is your [Y]%


In [10]:
wrong = test[test["llm_pred"] != test["intent"]]
print("Claude got", len(wrong), "wrong. Most common mix-ups:")
wrong.groupby(["intent", "llm_pred"]).size().sort_values(ascending=False).head(10)

Claude got 32 wrong. Most common mix-ups:


intent                    llm_pred             
contact_customer_service  contact_human_agent      7
track_refund              get_refund               5
check_invoice             get_invoice              3
delivery_options          place_order              3
cancel_order              payment_issue            2
check_refund_policy       delivery_period          2
delivery_period           track_order              2
get_invoice               check_invoice            2
change_order              cancel_order             1
create_account            registration_problems    1
dtype: int64

In [11]:
rows = []
for t in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    auto = test[test["confidence"] >= t]
    coverage = len(auto) / len(test) * 100
    precision = (auto["llm_pred"] == auto["intent"]).mean() * 100 if len(auto) else 0
    rows.append({"threshold": t, "auto_routed_%": round(coverage, 1), "precision_%": round(precision, 1)})
thresholds = pd.DataFrame(rows)
print("Pick the LOWEST threshold where precision_% is 95 or more.")
print("Its auto_routed_% is your [Z]%.")
thresholds

Pick the LOWEST threshold where precision_% is 95 or more.
Its auto_routed_% is your [Z]%.


,threshold,auto_routed_%,precision_%
0,0.50,99.8,93.8
1,0.60,99.2,94.4
2,0.70,97.6,95.1
3,0.80,96.8,95.5
4,0.90,86.4,98.1
5,0.95,37.4,100.0


In [12]:
PRICE_IN = 0.10     # US dollars per 1 million input tokens (Claude Haiku 5.5, Oct 2026)
PRICE_OUT = 0.50    # US dollars per 1 million output tokens
USD_TO_INR = 88     # change this to today's rate

cost_usd = (test["input_tokens"].sum() * PRICE_IN + test["output_tokens"].sum() * PRICE_OUT) / 1_000_000
per_1000_inr = cost_usd / len(test) * 1000 * USD_TO_INR
print(f"Total test cost: ${cost_usd:.4f}")
print(f"Cost per 1,000 tickets: Rs {per_1000_inr:.2f}   <-- this is your Rs [__]")

Total test cost: $0.1383
Cost per 1,000 tickets: Rs 24.34   <-- this is your Rs [__]


In [13]:
test.sample(50, random_state=1)[["instruction", "urgency"]].to_csv("urgency_check.csv", index=False)
print("Open urgency_check.csv, add a column 'agree' with yes/no, and count the yes.")

Open urgency_check.csv, add a column 'agree' with yes/no, and count the yes.


In [14]:
print("=== COPY THESE INTO YOUR CV ===")
print(f"Test set size: {len(test)}")
print(f"[X] Claude accuracy: {llm_acc:.1f}%")
print(f"[Y] Keyword baseline accuracy: {baseline_acc:.1f}%")
print(thresholds.to_string(index=False))
print(f"Cost per 1,000 tickets: Rs {per_1000_inr:.2f}")

=== COPY THESE INTO YOUR CV ===
Test set size: 500
[X] Claude accuracy: 93.6%
[Y] Keyword baseline accuracy: 44.0%
 threshold  auto_routed_%  precision_%
      0.50           99.8         93.8
      0.60           99.2         94.4
      0.70           97.6         95.1
      0.80           96.8         95.5
      0.90           86.4         98.1
      0.95           37.4        100.0
Cost per 1,000 tickets: Rs 24.34
